<a href="https://colab.research.google.com/github/tharindu33333/ME421-Mechanical-Systems-Lab-A2/blob/main/Vibration/%20E_20_222_Vibration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import scipy as sp
from scipy.integrate import odeint
import math
from numpy import linalg
import sympy

from sympy import symbols
from sympy import *
from sympy.physics.mechanics import dynamicsymbols, init_vprinting
init_vprinting(pretty_print=True)

import plotly.graph_objects as go

# Python Rigidbody Class Definition

# Activity 1 ( Derive the 2-dof freedom model that will capture the first two dominant modes of small amplitude vibration of the system.)

# # 1. Physical Description of the System

The TecQuipment vibration analyser consists of:

  A rigid beam with one end pinned and the other end connected to the frame through a spring

  An unbalanced motor mounted at the mid-span of the beam

  Small-amplitude transverse vibration measured at the motor location

  External disturbances transmitted through the base

  High flexural rigidity of the beam

Since the vibration amplitudes are small, the system can be assumed linear, and due to symmetry and stiffness, the vibration response is dominated by the first two bending modes.

---

## 2. Choice of Generalized Coordinates

To capture the dominant dynamics without detailed beam properties, the system is modeled using **modal coordinates**.

Let:

* $q_1(t)$: generalized coordinate of the **first dominant mode**
* $q_2(t)$: generalized coordinate of the **second dominant mode**

The transverse displacement at the motor (mid-span) is approximated as:

$$y(t) = \alpha_1 q_1(t) + \alpha_2 q_2(t)$$

where $\alpha_1$ and $\alpha_2$ are modal participation factors accounting for sensor location and mode shape influence.

---

## 3. Assumptions for the Model

Small-amplitude vibration → linear behavior

Light damping

Modal orthogonality → negligible modal coupling

Beam properties are not explicitly required (experimentally identified dynamics)

Under these assumptions, each mode behaves as an independent second-order system.

---
## 4. Equations of Motion (Modal Form)

The equations governing the first two modes are:

$$\ddot{q}_1 + 2\zeta_1\omega_1\dot{q}_1 + \omega_1^2q_1 = 0$$
$$\ddot{q}_2 + 2\zeta_2\omega_2\dot{q}_2 + \omega_2^2q_2 = 0$$

where:

* $\omega_1, \omega_2$: natural frequencies of the first and second dominant modes
* $\zeta_1, \zeta_2$: corresponding damping ratios

These parameters are identified experimentally from resonance and decay measurements.

---
## 5. State-Space Representation (2-DOF Model)

Define the state vector:

$$X = [q_1 \quad \dot{q}_1 \quad q_2 \quad \dot{q}_2]^T$$

The state-space form of the system becomes:

$$\dot{X} = \begin{bmatrix} 0 & 1 & 0 & 0 \\ -\omega_1^2 & -2\zeta_1\omega_1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & -\omega_2^2 & -2\zeta_2\omega_2 \end{bmatrix} X$$

This represents a **two-degree-of-freedom linear vibration model** capturing the first two dominant modes.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import math

class MugasRigidBodyFunctions:

    def __init__(self):
        pass


    def hat_matrix(self, x):
        """
        Computes the skew-symmetric matrix (hat matrix) corresponding to a 3 by 1 matrix.

        The hat matrix is a representation of the cross product operation as a matrix.
        It is commonly used in rigid body dynamics, robotics, and control theory to
        transform a vector into its corresponding cross-product operation.

        Args:
            x (numpy.ndarray): A 3-element numpy array representing a 3 by 1 matrix.

        Returns:
            numpy.ndarray: A 3x3 skew-symmetric matrix corresponding to the input 3 by 1 matrix.

        Example:
            Input: x = [1, 2, 3]
            Output:
            [[  0., -3.,  2.],
            [  3.,  0., -1.],
            [ -2.,  1.,  0.]]
        """
        return np.array([
            [0., -x[2], x[1]],
            [x[2],  0., -x[0]],
            [-x[1], x[0],  0.]
        ])


    def q_from_axis_angles(self, theta, unit_axis):
        """
        Computes a quaternion from a given rotation angle and unit axis.

        A quaternion is a compact representation of rotations in 3D space.
        It is composed of a scalar part and a 3 by 1 matrix part, where the scalar represents the
        rotation magnitude, and the 3 by 1 matrix encodes the rotation axis.

        Args:
            theta (float): The rotation angle in radians.
            unit_axis (list or numpy.ndarray): A 3-element unit 3 by 1 matrix representing the axis of rotation.

        Returns:
            numpy.ndarray: A 4-element array representing the quaternion [q0, q1, q2, q3],
                          where q0 is the scalar part, and [q1, q2, q3] is the 3 by 1 matrix part.

        Formula:
            Quaternion (q) = [cos(theta / 2), sin(theta / 2) * unit_axis]

        Example:
            Input: theta = π/2, unit_axis = [0, 0, 1] (z-axis)
            Output: [0.7071, 0.0, 0.0, 0.7071] (represents a 90-degree rotation about the z-axis)

        Notes:
            - Ensure the input `unit_axis` is normalized to avoid incorrect results.
            - Commonly used in 3D transformations and rigid body dynamics.

        """
        # Compute the scalar part of the quaternion (cosine of half the rotation angle)
        scalar_part = np.cos(theta / 2)

        # Compute the vector part of the quaternion (sine of half the rotation angle times the unit axis)
        three_by_one_matrix_part = np.sin(theta / 2) * np.array(unit_axis)

        # Combine scalar and vector parts into a single quaternion
        return np.concatenate(([scalar_part], three_by_one_matrix_part))


    def r_from_quaternions(self, q):
        """
        Computes a rotation matrix from a given quaternion.

        The rotation matrix is a \(3 \times 3\) orthogonal matrix that represents a rotation
        in 3D space. This method utilizes a quaternion to derive the corresponding rotation matrix.

        Args:
            q (numpy.ndarray): A 4-element numpy array representing the quaternion [q0, q1, q2, q3],
                              where q0 is the scalar part, and [q1, q2, q3] forms a \(3 \times 1\) matrix.

        Returns:
            numpy.ndarray: A \(3 \times 3\) rotation matrix corresponding to the input quaternion.

        Formula:
            Rotation Matrix (R) = I + 2 * q0 * hat(w) + 2 * hat(w) @ hat(w)
            - \(I\): Identity matrix (\(3 \times 3\)).
            - \(q0\): Scalar part of the quaternion.
            - \(w\): \(3 \times 1\) matrix (vector part of the quaternion).
            - \(hat(w)\): Skew-symmetric (hat) matrix of \(w\).

        Example:
            Input: q = [0.7071, 0.7071, 0, 0] (90-degree rotation about x-axis)
            Output:
            [[ 1.   0.   0.  ]
            [ 0.   0.  -1.  ]
            [ 0.   1.   0.  ]]

        Notes:
            - Input quaternion \(q\) should be normalized to ensure a valid rotation matrix.
            - The method converts the vector part of the quaternion into a \(3 \times 1\) matrix internally.
        """
        # Extract the scalar part of the quaternion (q0)
        q0 = q[0]

        # Extract the direction part and reshape it into a 3x1 matrix
        w = q[1:] #.reshape((3, 1))

        # Compute the rotation matrix using the quaternion formula
        return (
            np.identity(3) +
            2 * q0 * self.hat_matrix(w) +
            2 * self.hat_matrix(w) @ self.hat_matrix(w)
        )


    def rotation_matrix_2_euler_angles(self, R):
        """
        Converts a \(3 \times 3\) rotation matrix into its corresponding Euler angles.

        Euler angles provide a representation of rotations using three consecutive rotations about
        specified axes. This method extracts the angles (φ, θ, ψ) from the rotation matrix \(R\).

        Args:
            R (numpy.ndarray): A \(3 \times 3\) rotation matrix representing the rotation in 3D space.

        Returns:
            tuple: A tuple of three angles (φ, θ, ψ) in radians:
                - φ (phi): Rotation about the z-axis.
                - θ (theta): Rotation about the y-axis.
                - ψ (psi): Rotation about the x-axis.

        Formula:
            Depending on the structure of the rotation matrix \(R\):
            - Handle standard cases when \(R[2, 2] > -1\) and \(R[2, 2] < 1\).
            - Handle gimbal lock cases when \(R[2, 2] = ±1\).

        Example:
            Input:
            R = [[ 0.866, -0.5,  0. ],
                [ 0.5,    0.866, 0. ],
                [ 0. ,    0. ,   1. ]]
            Output: (φ=0.5236, θ=0.0, ψ=0.0) (30° rotation about z-axis)

        Notes:
            - Euler angles are dependent on the chosen axis order (assumes ZYX order here).
            - Handles edge cases like gimbal lock (singularities in rotation representation).
        """
        # Check the third row, third column element to determine the rotation scenario
        if R[2, 2] < 1:  # General case: not gimbal lock
            if R[2, 2] > -1:  # Unique solution
                phi = np.pi - math.atan2(R[0, 2], R[1, 2])  # Rotation about z-axis
                theta = math.acos(R[2, 2])  # Rotation about y-axis
                psi = np.pi - math.atan2(R[2, 0], -R[2, 1])  # Rotation about x-axis
                return phi, theta, psi
            # Gimbal lock: theta = π
            phi = -math.atan2(R[0, 1], -R[0, 0])  # Rotation about z-axis
            return phi, np.pi, 0
        # Gimbal lock: theta = 0
        phi = math.atan2(R[0, 1], R[0, 0])  # Rotation about z-axis
        return phi, 0, 0

    def re3_equals_gamma(self, gamma):
        """
        Computes a rotation matrix \( R \) such that \( R \cdot e_3 = \gamma \),
        where \( e_3 \) is the unit axis along the z-axis.

        This method constructs a rotation matrix that aligns the z-axis (\( e_3 \))
        with a given 3D vector (\( \gamma \)). It uses the axis-angle representation
        to calculate the required rotation.

        Args:
            gamma (numpy.ndarray): A 3-element numpy array (or \(3 \times 1\) matrix)
                                  representing the target axis to align \( e_3 \) with.
                                  \( \gamma \) should be normalized.

        Returns:
            numpy.ndarray: A \(3 \times 3\) rotation matrix \( R \) such that \( R \cdot e_3 = \gamma \).

        Formula:
            1. Compute the angle \( \theta = \arccos(\gamma[2]) \).
            2. Determine the rotation axis \( n \):
              \( n = [-\gamma[1]/\sin(\theta), \gamma[0]/\sin(\theta), 0] \).
            3. Construct the quaternion using \( \theta \) and \( n \).
            4. Convert the quaternion to a rotation matrix \( R \).

        Example:
            Input:
            gamma = [0, 0, 1]  # Already aligned with e_3
            Output:
            [[1.0, 0.0, 0.0],
            [0.0, 1.0, 0.0],
            [0.0, 0.0, 1.0]]  # Identity matrix

            Input:
            gamma = [1/√2, 0, 1/√2]  # 45-degree tilt in the xz-plane
            Output:
            [[ 0.7071, 0.0,  0.7071],
            [ 0.0,    1.0,  0.0   ],
            [-0.7071, 0.0,  0.7071]]

        Notes:
            - \( \gamma \) must be normalized before passing to the function.
            - If \( \sin(\theta) \) is close to zero, the method handles the edge case
              where \( \gamma \) is aligned with \( e_3 \) directly.
        """
        # Compute the rotation angle (theta) between e3 and gamma
        theta = math.acos(gamma[2])

        # Handle edge case where gamma is already aligned with e3
        if np.isclose(np.sin(theta), 0):
            return np.identity(3)  # No rotation needed

        # Compute the rotation axis (n) as a 3x1 matrix
        n = np.array([[-gamma[1] / np.sin(theta)],
                      [gamma[0] / np.sin(theta)],
                      [0]])

        # Construct the quaternion and compute the rotation matrix
        return self.r_from_quaternions(self.q_from_axis_angles(theta, n))


    def rotate_and_translate(self, object_vertices, R, o):
        """
        Applies a rotation and translation to a set of object vertices.

        This method transforms an object's vertices by first applying a rotation
        using the provided rotation matrix \( R \), and then translating the result
        by the translation  \( o \).

        Args:
            object_vertices (numpy.ndarray): A \(3 \times N\) matrix where each column represents
                                            the coordinates of a vertex in the object.
            R (numpy.ndarray): A \(3 \times 3\) rotation matrix representing the orientation.
            o (numpy.ndarray): A \(3 \times 1\) matrix (or list/array of 3 elements)
                              representing the translation.

        Returns:
            numpy.ndarray: A \(3 \times N\) matrix where each column represents the transformed
                          coordinates of the vertices after applying the rotation and translation.

        Formula:
            Transformed Vertices = Translation + Rotation @ Original Vertices
            \[
            \text{Transformed Vertices} = o + R \cdot \text{object\_vertices}
            \]

        Example:
            Input:
            object_vertices = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]  # 3 vertices
            R = [[0, -1, 0], [1, 0, 0], [0, 0, 1]]  # 90-degree rotation about z-axis
            o = [1, 1, 0]  # Translation]

            Output:
            [[1, 1, 1], [0, 2, 1], [0, 0, 1]]  # Transformed vertices

        Notes:
            - The object vertices should be provided as a \(3 \times N\) matrix.
            - The translation  \( o \) should be a \(3 \times 1\) matrix or convertible to one.
            - Ensures seamless transformation of 3D objects in simulations and graphics.
        """
        # Reshape the translation into a 3x1 matrix if not already in that form
        o = np.array([[o[0], o[1], o[2]]]).T

        # Apply the rotation and translation to the object vertices
        return o + R @ object_vertices

    def add_orth_norm_frame(self, fig, o, R, axis_range, axis_color):
        """
        Adds an orthonormal frame to a 3D Plotly figure, representing the axes of a rotated frame.

        This method visualizes a rotated orthonormal frame in a 3D plot. The frame is defined
        by its origin and a rotation matrix, and it is represented by three arrows indicating
        the rotated \( x \)-axis, \( y \)-axis, and \( z \)-axis.

        Args:
            fig (plotly.graph_objects.Figure): The 3D figure to which the frame will be added.
            o (numpy.ndarray): A \(3 \times 1\) matrix (or list/array of 3 elements) representing
                              the origin of the frame in 3D space.
            R (numpy.ndarray): A \(3 \times 3\) rotation matrix defining the orientation of the frame.
            axis_range (list of tuples): Specifies the range for each axis in the format
                                        [(x_min, x_max), (y_min, y_max), (z_min, z_max)].
            axis_color (str): Color for the axis lines in the frame (e.g., "red", "blue", "green").

        Returns:
            plotly.graph_objects.Figure: The input figure with the added orthonormal frame.

        Example:
            Input:
            fig = go.Figure()  # Empty 3D figure
            o = [0, 0, 0]  # Origin at (0, 0, 0)
            R = [[0, -1, 0], [1, 0, 0], [0, 0, 1]]  # 90-degree rotation about z-axis
            axis_range = [(-1, 1), (-1, 1), (-1, 1)]
            axis_color = "blue"

            Output:
            A 3D figure with the rotated frame visualized.

        Notes:
            - Each axis of the frame is scaled based on the rotation matrix \( R \).
            - The figure layout is updated to match the specified axis range and ensure aspect ratio consistency.
        """
        # Define the standard basis vectors for the x, y, and z axes
        e = [np.array([1, 0, 0]), np.array([0, 1, 0]), np.array([0, 0, 1])]

        # Apply the rotation matrix R to each e-frame axis to compute the rotated b-frame axes
        b = [R @ ei for ei in e]

        # Add each axis as a line starting at origin `o` and extending in the rotated direction
        for bi in b:
            fig.add_trace(go.Scatter3d(
                x=[o[0], o[0] + bi[0]],  # Line along the rotated axis
                y=[o[1], o[1] + bi[1]],
                z=[o[2], o[2] + bi[2]],
                hoverinfo='x+y+z',  # Tooltip displays the 3D coordinates
                mode='lines',  # Display as lines
                line=dict(width=8, color=axis_color)  # Line styling
            ))

        # Update the layout to fix the axis ranges and maintain aspect ratio
        fig.update_layout(
            showlegend=False,  # Hide legend
            scene=dict(
                xaxis=dict(range=axis_range[0], autorange=False),
                yaxis=dict(range=axis_range[1], autorange=False),
                zaxis=dict(range=axis_range[2], autorange=False),
                aspectratio=dict(x=1, y=1, z=1)  # Keep the aspect ratio uniform
            )
        )
        return fig

    def add_orth_norm_frame(self, fig, o, R, axis_range, axis_color):
        """
        Adds an orthonormal frame to a 3D Plotly figure, representing the axes of a rotated frame.

        This method visualizes a rotated orthonormal frame in a 3D plot. The frame is defined
        by its origin and a rotation matrix, and it is represented by three arrows indicating
        the rotated \( x \)-axis, \( y \)-axis, and \( z \)-axis.

        Args:
            fig (plotly.graph_objects.Figure): The 3D figure to which the frame will be added.
            o (numpy.ndarray): A \(3 \times 1\) matrix (or list/array of 3 elements) representing
                              the origin of the frame in 3D space.
            R (numpy.ndarray): A \(3 \times 3\) rotation matrix defining the orientation of the frame.
            axis_range (list of tuples): Specifies the range for each axis in the format
                                        [(x_min, x_max), (y_min, y_max), (z_min, z_max)].
            axis_color (str): Color for the axis lines in the frame (e.g., "red", "blue", "green").

        Returns:
            plotly.graph_objects.Figure: The input figure with the added orthonormal frame.

        Example:
            Input:
            fig = go.Figure()  # Empty 3D figure
            o = [0, 0, 0]  # Origin at (0, 0, 0)
            R = [[0, -1, 0], [1, 0, 0], [0, 0, 1]]  # 90-degree rotation about z-axis
            axis_range = [(-1, 1), (-1, 1), (-1, 1)]
            axis_color = "blue"

            Output:
            A 3D figure with the rotated frame visualized.

        Notes:
            - Each axis of the frame is scaled based on the rotation matrix \( R \).
            - The figure layout is updated to match the specified axis range and ensure aspect ratio consistency.
        """
        # Define the e-frame axis for the x, y, and z axes
        e = [np.array([1, 0, 0]), np.array([0, 1, 0]), np.array([0, 0, 1])]

        # Apply the rotation matrix R to each e-frame axis to compute the rotated b-frame axes
        b = [R @ ei for ei in e]

        # Add each axis as a line starting at origin `o` and extending in the rotated direction
        for bi in b:
            fig.add_trace(go.Scatter3d(
                x=[o[0], o[0] + bi[0]],  # Line along the rotated axis
                y=[o[1], o[1] + bi[1]],
                z=[o[2], o[2] + bi[2]],
                hoverinfo='x+y+z',  # Tooltip displays the 3D coordinates
                mode='lines',  # Display as lines
                line=dict(width=8, color=axis_color)  # Line styling
            ))

        # Update the layout to fix the axis ranges and maintain aspect ratio
        fig.update_layout(
            showlegend=False,  # Hide legend
            scene=dict(
                xaxis=dict(range=axis_range[0], autorange=False),
                yaxis=dict(range=axis_range[1], autorange=False),
                zaxis=dict(range=axis_range[2], autorange=False),
                aspectratio=dict(x=1, y=1, z=1)  # Keep the aspect ratio uniform
            )
        )
        return fig

    def animate_particle_motion(self, xx, axis_range, fig_title):
        """
        Creates a 3D animated visualization of a particle's motion over time.

        This method visualizes the trajectory of a particle in 3D space, showing both
        the particle's current position as a marker and the entire path it follows as a line.

        Args:
            xx (list of tuples): A list of 3D coordinates representing the particle's position
                                at each time step, formatted as [(x1, y1, z1), (x2, y2, z2), ...].
            axis_range (list of tuples): Specifies the range for each axis in the format
                                        [(x_min, x_max), (y_min, y_max), (z_min, z_max)].
            fig_title (str): Title for the animated 3D plot.

        Returns:
            plotly.graph_objects.Figure: A Plotly figure object containing the animated 3D scatter plot.

        Example:
            Input:
            xx = [(0, 0, 0), (1, 0, 0), (2, 1, 0), (3, 2, 1)]  # Particle's path
            axis_range = [(-5, 5), (-5, 5), (-5, 5)]  # Axis limits
            fig_title = "Particle Motion Animation"

            Output:
            A Plotly animated 3D scatter plot showing the particle's motion over time.

        Notes:
            - The particle's current position is shown as a red marker.
            - The complete trajectory is shown as a blue line.
            - The animation can be controlled using play/pause buttons in the Plotly interface.

        """
        # Unpack the particle's trajectory into separate x, y, z coordinates
        x_vals, y_vals, z_vals = zip(*xx)

        # Define the initial position of the particle as a red marker
        trace_particle = go.Scatter3d(
            x=[x_vals[0]], y=[y_vals[0]], z=[z_vals[0]],  # Start with the first position
            mode="markers",  # Show as a marker
            marker=dict(color="red", size=10)  # Red marker with size 10
        )

        # Define the complete trajectory as a blue line
        trace_path = go.Scatter3d(
            x=x_vals, y=y_vals, z=z_vals,  # All trajectory points
            mode="lines",  # Show as a line
            line=dict(color="blue", width=2),  # Blue line with width 2
            name='Path'
        )

        # Define the layout of the plot
        layout = go.Layout(
            title_text=fig_title,  # Title of the figure
            hovermode="closest",  # Tooltip shows closest data point
            updatemenus=[dict(
                type="buttons",
                buttons=[dict(
                    label="Play",  # Play button for the animation
                    method="animate",
                    args=[None]
                )]
            )],
            scene=dict(  # 3D scene settings
                xaxis=dict(range=axis_range[0], autorange=False),  # Fixed x-axis range
                yaxis=dict(range=axis_range[1], autorange=False),  # Fixed y-axis range
                zaxis=dict(range=axis_range[2], autorange=False),  # Fixed z-axis range
                aspectratio=dict(x=1, y=1, z=1)  # Uniform aspect ratio
            )
        )

        # Create animation frames for each point in the trajectory
        frames = [go.Frame(
            data=[go.Scatter3d(
                x=[point[0]], y=[point[1]], z=[point[2]],  # Current position of the particle
                mode="markers",  # Show as a marker
                marker=dict(color="red", size=10),  # Red marker with size 10
                name='Particle'
            )]) for point in xx]

        # Create the Plotly figure with initial data, layout, and frames
        fig = go.Figure(data=[trace_particle, trace_path], layout=layout, frames=frames)

        # Display the figure
        fig.show()

        # Return the figure object
        return fig

    def rigid_body_system(self, parameters, t, X):
        """
        Models the dynamics of a rigid body system.

        This method computes the time derivatives and other intermediate quantities
        for a rigid body's state, based on the provided parameters and external forces/torques.

        Args:
            parameters (dict): A dictionary containing parameters for the rigid body system:
                - 'CM' (numpy.ndarray): Center of mass position as a \(3 \times 1\) matrix.
                - 'M' (float): Mass of the rigid body.
            t (float): Current time (useful for time-dependent forces/torques).
            X (list): State of the system, containing:
                - \( X[0][0] \) (numpy.ndarray): Rotation matrix \( R \) (\(3 \times 3\)).
                - \( X[1] \) (numpy.ndarray): Spatial angular velocity \( \omega \) (\(3 \times 1\)).
                - \( X[2] \) (numpy.ndarray): Linear momentum r \( p \) (\(3 \times 1\)).

        Returns:
            list: A list of quantities for the rigid body dynamics:
                - \( \thetaomega \) (float): Magnitude of spatial angular velocity \( \omega \).
                - \( \nomega \) (numpy.ndarray): Normalized spatial angular velocity axis(\(3 \times 1\)).
                - \( \doto \) (numpy.ndarray): Time derivative of the position (\(3 \times 1\)).
                - \( dp \) (numpy.ndarray): Time derivative of linear momentum (\(3 \times 1\)).
                - \( dspi \) (numpy.ndarray): Time derivative of spatial angular momentum (\(3 \times 1\)).
                - \( dXc \) (numpy.ndarray): Time derivative of other external dynamics (\(3 \times 1\)).

        Formula:
            The equations are modeled as:
            - \( \dot{o} = \frac{p}{M} \)
            - \( \dot{p} = f_e + f_a \)
            - \( \dot{\text{spi}} = \tau_e + \tau_a \)
            - \( \omega = \text{angular velocity} \)

        Example:
            Input:
            parameters = {'CM': [0, 0, 0], 'M': 10}
            t = 0
            X = [[[np.eye(3)], [1, 0, 0], [0, 1, 0]]]

            Output:
            - \( \thetaomega \): 1.0
            - \( \nomega \): [1, 0, 0]
            - \( \doto \): [0, 0.1, 0.1]
            - \( dp \): ...
            - ...

        Notes:
            - Requires user-defined `externalForceModel` and `actuator` functions to provide
              external and actuator forces/torques.
            - Spatial angular velocity \( \omega \) is normalized if its magnitude exceeds a small threshold.

        """
        # Extract parameters
        barX, M = parameters['CM'], parameters['M']

        # Extract state variables
        R = X[0][0]  # Rotation matrix
        omega = X[1]  # Spatial angular velocity
        p = X[2]  # Linear momentum

        # Compute external and actuator forces and torques
        taue, fe = externalForceModel(self, parameters, X)  # External forces and torques
        taua, fa = actuator(self, parameters, t, X, taue, fe)  # Actuator forces and torques

        # Compute time derivatives of position, momentum, and spin
        doto = p / M  # Time derivative of position
        dp = fe + fa  # Time derivative of linear momentum
        dspi = taue + taua  # Time derivative of spin

        # External dynamics (can be expanded with a controller model)
        dXc = np.array([0., 0., 0.])

        # Compute angular velocity properties
        if np.linalg.norm(omega) >= 0.0001:  # Avoid division by zero
            nomega = omega / np.linalg.norm(omega)  # Normalized angular velocity
            thetaomega = np.linalg.norm(omega)  # Magnitude of angular velocity
        else:  # Handle case of negligible angular velocity
            nomega = np.array([0, 0, 0])
            thetaomega = 0

        # Return computed quantities
        return [thetaomega, nomega, doto, dp, dspi, dXc]


    def animate_2D_scatter_plot(self, x, YY, xlabel, ylabel, title):
        """
        Creates an animated 2D scatter plot using Plotly.

        This method visualizes a series of 2D data points as an animation, where the y-values
        evolve over time for a fixed set of x-values. It is useful for illustrating dynamic
        changes in data over time.

        Args:
            x (numpy.ndarray): A 1D array representing the x-axis values.
            YY (numpy.ndarray): A 2D array where each row represents the y-values at a specific
                                time step, and columns correspond to the x-values.
            xlabel (str): Label for the x-axis.
            ylabel (str): Label for the y-axis.
            title (str): Title for the animated plot.

        Returns:
            plotly.graph_objects.Figure: A Plotly figure object containing the animated scatter plot.

        Example:
            Input:
            x = np.linspace(0, 10, 100)  # x-values
            YY = np.array([np.sin(x + t) for t in np.linspace(0, 2 * np.pi, 50)])  # y-values evolve over time
            xlabel = "X-axis"
            ylabel = "Y-axis"
            title = "Animated 2D Scatter Plot"

            Output:
            A Plotly animated scatter plot showing the evolution of the sine wave over time.

        Notes:
            - The animation buttons are included by default in the plot layout.
            - The range of the y-axis is automatically determined based on the data in `YY`.

        """
        # Define the layout of the plot
        layout = go.Layout(
            xaxis={'title': xlabel},  # Label for the x-axis
            yaxis={'title': ylabel, 'range': [1.1 * YY.min(), 1.1 * YY.max()]},  # Label and range for the y-axis
            title={'text': title, 'y': 0.9, 'x': 0.5, 'xanchor': 'center', 'yanchor': 'top'},  # Title styling
            scene=dict(aspectratio=dict(x=1, y=1)),  # Maintain aspect ratio
            hovermode="closest",  # Display closest data point information
            updatemenus=[dict(
                type="buttons",
                buttons=[
                    dict(
                        label="Play",  # Animation play button
                        method="animate",
                        args=[None]
                    )
                ]
            )]
        )

        # Create frames for each time step in YY
        frames = [go.Frame(data=[go.Scatter(x=x, y=y)]) for y in YY]

        # Initialize the figure with the first frame's data and the layout
        fig = go.Figure(data=[go.Scatter(x=x, y=YY[0, :])], layout=layout, frames=frames)

        # Return the figure object (can be displayed using fig.show())
        return fig


    def simulate_dy_system(self, dynamic_system_model, t_max, dt, x0, sys_para, fig_title, x_label, y_label):
        """
        Simulates the dynamics of a system using a numerical solver and visualizes the results.

        This method solves a system of differential equations defined by a user-provided
        dynamic system model, using initial conditions and system parameters. The solution
        is plotted over time for visualization.

        Args:
            dynamic_system_model (function): The function defining the system's dynamics.
                                            It should return \( \frac{dx}{dt} \) given
                                            the current state, time, and parameters.
            t_max (float): The total simulation time.
            dt (float): The time step for the simulation.
            x0 (numpy.ndarray): Initial state of the system (\( n \times 1 \) matrix or array).
            sys_para (any): Parameters for the dynamic system model, passed as an argument to the function.
            fig_title (str): Title for the generated plot.
            x_label (str): Label for the x-axis of the plot (typically time).
            y_label (str): Label for the y-axis of the plot (state variables).

        Returns:
            tuple: A tuple containing:
                - t (numpy.ndarray): The array of time points.
                - sol (numpy.ndarray): The solution matrix where each row corresponds to
                                      the state vector at a specific time.
                - fig (plotly.graph_objects.Figure): The Plotly figure object showing the simulation results.

        Example:
            Input:
            dynamic_system_model = LinearSystemModel  # A predefined dynamic model
            t_max = 10  # Simulate for 10 seconds
            dt = 0.1  # Time step of 0.1 seconds
            x0 = [1, 0]  # Initial state
            sys_para = [[0, 1], [-1, -2]]  # System matrix A
            fig_title = "System Dynamics"
            x_label = "Time (s)"
            y_label = "State Variables"

            Output:
            - Time array \( t \)
            - Solution matrix \( sol \)
            - Plotly figure showing the state variables over time

        Notes:
            - The `dynamic_system_model` must accept parameters in the form
              `(state_vector, time, system_parameters)`.
            - Uses `scipy.integrate.odeint` for numerical integration.
            - Visualization is done using Plotly, with each state variable plotted as a line.

        """
        # Generate time points for the simulation
        t = np.linspace(0, t_max, int(t_max / dt + 1))

        # Solve the system of differential equations
        sol = odeint(dynamic_system_model, x0, t, args=(sys_para,))

        # Create a Plotly figure for visualization
        fig = go.Figure()

        # Add a line plot for each state variable
        for sol_col in sol.T:
            fig.add_trace(go.Scatter(x=t, y=sol_col, mode='lines+markers'))

        # Update the figure layout with titles and axis labels
        fig.update_layout(
            title=fig_title,
            xaxis=dict(title=x_label),
            yaxis=dict(title=y_label)
        )

        # Display the figure
        fig.show()

        # Return the time points, solution matrix, and figure
        return t, sol, fig

    def LinearSystemModel(self, X, t, A):
        """
        Represents a linear dynamic system model.

        This function defines the evolution of a linear system over time using the
        state-space representation. It calculates the time derivative of the state
         \( X \) given the system matrix \( A \).

        Args:
            X (numpy.ndarray): A \( n \times 1 \) matrix (or list/array of \( n \) elements)
                              representing the state of the system.
            t (float): Time variable (not used in the calculation but required for compatibility
                      with ODE solvers).
            A (numpy.ndarray): A \( n \times n \) matrix representing the system matrix that defines
                              the linear dynamics.

        Returns:
            numpy.ndarray: The time derivative of the state \( dX/dt \), computed as \( A \cdot X \).

        Formula:
            The system is modeled as:
            \[
            \frac{dX}{dt} = A \cdot X
            \]

        Example:
            Input:
            X = [1, 2]
            t = 0  # Time (not used in this example)
            A = [[0, 1], [-1, -2]]  # System matrix

            Output:
            [-2, -5]  # Time derivative of the state vector

        Notes:
            - Ensure the dimensions of \( A \) and \( X \) are consistent (\( A \) should be square and
              \( X \) should have the appropriate size).
            - Commonly used in simulations of linear systems such as control systems and electrical circuits.
        """
        # Compute the time derivative of the state vector using the system matrix
        dXdt = A @ X
        return dXdt


    def cube_vertices(self, cube_dimensions):
        l, w, h = cube_dimensions['l'], cube_dimensions['w'], cube_dimensions['h']
        xp, yp, zp = cube_dimensions['xp'], cube_dimensions['yp'], cube_dimensions['zp']

        X = [-xp, -xp, l-xp, l-xp, -xp, -xp, l-xp, l-xp]
        Y = [-yp, w-yp, w-yp, -yp, -yp, w-yp, w-yp, -yp]
        Z = [-zp, -zp, -zp, -zp, h-zp, h-zp, h-zp, h-zp]

        return [X, Y, Z]

    def animated_cube_flat_shading(self, cubeVertices,figTitle):
        fig = go.Figure(
            frames=[go.Frame(data=[
            go.Mesh3d(
                # 8 vertices of a cube
                x=xx[0][0],
                y=xx[0][1],
                z=xx[0][2],
                # i, j and k give the vertices of triangles
                i = [7, 0, 0, 0, 4, 4, 6, 6, 4, 0, 3, 2],
                j = [3, 4, 1, 2, 5, 6, 5, 2, 0, 1, 6, 3],
                k = [0, 7, 2, 3, 6, 7, 1, 1, 5, 5, 7, 6],
                name='y',
                opacity=0.6,
                color='#DC143C',
                flatshading = True)]) for xx in cubeVertices])

        fig.add_trace(go.Mesh3d(
                # 8 vertices of a cube
                x=cubeVertices[0][0][0],
                y=cubeVertices[0][0][1],
                z=cubeVertices[0][0][2],
                # i, j and k give the vertices of triangles
                i = [7, 0, 0, 0, 4, 4, 6, 6, 4, 0, 3, 2],
                j = [3, 4, 1, 2, 5, 6, 5, 2, 0, 1, 6, 3],
                k = [0, 7, 2, 3, 6, 7, 1, 1, 5, 5, 7, 6],
                name='y',
                opacity=0.6,
                color='#DC143C',
                flatshading = True)
            )

        duration=10;
        fig.update_layout(
            title=figTitle,width=600,height=600,
            scene=dict(xaxis=dict(range=[-5., 5.], autorange=False),yaxis=dict(range=[-5., 5.], autorange=False),zaxis=dict(range=[-5., 5.], autorange=False),aspectratio=dict(x=1, y=1, z=1),),
            updatemenus=[dict(type="buttons",
                                buttons=[dict(label="Play",
                                                method="animate",
                                                args=[None, {"frame": {"duration": duration},"mode": "immediate","fromcurrent": True, "transition": {"duration": duration, "easing": "linear"},}]
                                                )])])
        len(fig.frames)
        fig.show()
        return fig


    def eulers_method(self, dt, Tmax, parameters, ICs):
        M, II = parameters['M'], parameters['II']
        invII = np.linalg.inv(II)
        timeSteps = np.arange(0, Tmax+dt, dt)
        R, o, omega, doto, Xc = ICs[0][0], ICs[0][1], ICs[1], ICs[2], ICs[3]
        spi = R @ II @ R.T @ omega
        p = M * doto
        Xout = [ICs]

        for t in timeSteps:
            taue, fe = externalForceModel(self, parameters, X)
            taua, fa = actuator(self, parameters, t, X, taue, fe)

            dspi = taue + taua
            dp = fe + fa

            if np.linalg.norm(omega) >= 0.0001:
                nomega = omega / np.linalg.norm(omega)
                thetaomegat = dt * np.linalg.norm(omega)
                qomegat = np.concatenate(([np.cos(thetaomegat/2)], np.sin(thetaomegat/2) * nomega))
                R = self.r_from_quaternionsns(qomegat) @ R

            o += dt * doto
            spi += dt * dspi
            p += dt * dp
            doto = p / M
            omega = R @ invII @ R.T @ spi
            X = [[R, o], omega, doto, Xc]
            Xout.append(X)

        return Xout

    def runga_kutta_method(self, dt, Tmax, parameters, ICs):
        M, II = parameters['M'], parameters['II']
        invII = np.linalg.inv(II)
        timeSteps = np.arange(0, Tmax+dt, dt)
        X=ICs;
        Xout=[X];

        for t in timeSteps:
            Y1 = self.rk4_function(0.5*dt, X, t, X, parameters)
            Y2 = self.rk4_function(0.5*dt, X, t+0.5*dt, Y1, parameters)
            Y3 = self.rk4_function(dt, X, t+0.5*dt, Y2, parameters)

            values = [self.rigid_body_system(parameters, t+i*dt, X_j) for i, X_j in enumerate([X, Y1, Y2, Y3])]
            thetas, n_omegas, dotos, dps, dspis, dXcs = zip(*values)

            omegak = (dt/6.0) * sum(t * n for t, n in zip(thetas, n_omegas))
            nomegak = omegak/np.linalg.norm(omegak) if np.linalg.norm(omegak) >= 0.0001 else np.array([0, 0, 0])
            qomegak = np.concatenate(([np.cos(np.linalg.norm(omegak)/2)], np.sin(np.linalg.norm(omegak)/2) * nomegak))
            Rk = self.r_from_quaternions(qomegak) @ X[0][0]

            ok = X[0][1] + dt * np.mean(dotos)
            pk = X[2] + dt * np.mean(dps)
            spik = X[0][0] @ II @ X[0][0].T @ X[1] + dt * np.mean(dspis)
            Xck = X[3] + dt * np.mean(dXcs)

            omegak = Rk @ invII @ Rk.T @ spik
            X = [[Rk, ok], omegak, pk, Xck]
            Xout.append(X)

        return Xout

    def rk4_function(self, dtk, X, tk, Xk, parameters):
        M, II = parameters['M'], parameters['II']
        thetaomega1, nomega1, doto1, dp1, dspi1, dXc1 = self.rigid_body_system(parameters, tk, Xk)
        qomega1 = np.concatenate(([np.cos(dtk*thetaomega1/2)], np.sin(dtk*thetaomega1/2) * nomega1))
        R1 = self.r_from_quaternions(qomega1) @ X[0][0]
        p1 = X[2] + dtk * dp1
        spi1 = X[0][0] @ II @ X[0][0].T @ X[1] + dtk * dspi1
        omega1 = R1 @ np.linalg.inv(II) @ R1.T @ spi1
        X1 = [[R1, X[0][1] + dtk * doto1], omega1, p1, X[3] + dtk * dXc1]
        return X1


    def simulating_a_cube(self, dt, Tmax, cubeDimensions, parameters,ICs):
        XX=self.cube_vertices(cubeDimensions);

        #Xs=self.eulers_method(dt,Tmax,parameters,ICs);
        Xs=self.runga_kutta_method(dt,Tmax,parameters,ICs);
        ICR=ICs[0][0];
        XX0=ICR @ XX;

        rotatedVertices=[[XX0]]
        for X in Xs:
        #print(X[0])
            R=X[0][0];
            o=X[0][1];
            XXi=self.rotate_and_translate(XX,R,o);
            XX0=XXi;
            rotatedVertices+=[[XX0]];
        return rotatedVertices

<>:81: SyntaxWarning: invalid escape sequence '\('
<>:125: SyntaxWarning: invalid escape sequence '\('
<>:171: SyntaxWarning: invalid escape sequence '\('
<>:234: SyntaxWarning: invalid escape sequence '\('
<>:280: SyntaxWarning: invalid escape sequence '\('
<>:344: SyntaxWarning: invalid escape sequence '\('
<>:499: SyntaxWarning: invalid escape sequence '\('
<>:648: SyntaxWarning: invalid escape sequence '\('
<>:720: SyntaxWarning: invalid escape sequence '\('
<>:81: SyntaxWarning: invalid escape sequence '\('
<>:125: SyntaxWarning: invalid escape sequence '\('
<>:171: SyntaxWarning: invalid escape sequence '\('
<>:234: SyntaxWarning: invalid escape sequence '\('
<>:280: SyntaxWarning: invalid escape sequence '\('
<>:344: SyntaxWarning: invalid escape sequence '\('
<>:499: SyntaxWarning: invalid escape sequence '\('
<>:648: SyntaxWarning: invalid escape sequence '\('
<>:720: SyntaxWarning: invalid escape sequence '\('
/tmp/ipython-input-958190293.py:81: SyntaxWarning: invalid escape 

In [ ]:
import numpy as np

# ----------------------------------------------------------
# 1. CREATE CLASS OBJECT
# ----------------------------------------------------------
rb = MugasRigidBodyFunctions()

# ----------------------------------------------------------
# 2. EXPERIMENTALLY IDENTIFIED PARAMETERS
# (NOT beam material properties)
# ----------------------------------------------------------
f1 = 12.0     # First dominant frequency (Hz)
f2 = 35.0     # Second dominant frequency (Hz)

omega1 = 2 * np.pi * f1
omega2 = 2 * np.pi * f2

zeta1 = 0.03   # Light damping (experimentally realistic)
zeta2 = 0.02

# ----------------------------------------------------------
# 3. STATE-SPACE MATRIX (2-DOF MODAL MODEL)
# ----------------------------------------------------------
A_mat = np.array([
    [0, 1, 0, 0],
    [-omega1**2, -2*zeta1*omega1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, -omega2**2, -2*zeta2*omega2]
])

# ----------------------------------------------------------
# 4. INITIAL CONDITIONS (SMALL AMPLITUDE)
# ----------------------------------------------------------
x0 = np.array([
    4.3e-3,   # q1(0) -> measured large amplitude
    0.0,
    1.0e-3,   # q2(0)
    0.0
])

# ----------------------------------------------------------
# 5. SIMULATION USING YOUR CLASS
# ----------------------------------------------------------
t, sol, fig = rb.simulate_dy_system(
    dynamic_system_model=rb.LinearSystemModel,
    t_max=2.0,
    dt=0.0005,
    x0=x0,
    sys_para=A_mat,
    fig_title="2-DOF Small-Amplitude Vibration Model (TecQuipment Lab)",
    x_label="Time (s)",
    y_label="Modal Coordinates"
)

# ----------------------------------------------------------
# 6. RECONSTRUCT MEASURED MID-SPAN DISPLACEMENT
# ----------------------------------------------------------
# Mode participation factors (absorbed gains)
alpha1 = 1.0
alpha2 = 0.6

q1 = sol[:, 0]
q2 = sol[:, 2]

y_mid = alpha1 * q1 + alpha2 * q2

# ----------------------------------------------------------
# 7. VISUALIZE MID-SPAN RESPONSE
# ----------------------------------------------------------
rb.animate_2D_scatter_plot(
    x=t,
    YY=y_mid.reshape(1, -1),
    xlabel="Time (s)",
    ylabel="Mid-span displacement (m)",
    title="Measured Displacement Reconstruction at Motor Location"
)


# Activity 1 ( Derive the 2-dof freedom model that will capture the first two dominant modes of small amplitude vibration of the system.)

# Activity 2 ( Estimate the damping ratios of the first two dominant modes. )

In [ ]:
def estimate_damping_log_dec(peaks):
    """
    Estimate damping ratio using logarithmic decrement
    """
    delta = np.mean(np.log(peaks[:-1] / peaks[1:]))
    zeta = delta / np.sqrt(4 * np.pi**2 + delta**2)
    return zeta


In [ ]:
# Modal responses from previous simulation
q1 = sol[:, 0]   # Mode 1 coordinate
q2 = sol[:, 2]   # Mode 2 coordinate


In [ ]:
def extract_peaks(signal, threshold=1e-4):
    peaks = []
    for i in range(1, len(signal)-1):
        if signal[i] > signal[i-1] and signal[i] > signal[i+1]:
            if abs(signal[i]) > threshold:
                peaks.append(abs(signal[i]))
    return np.array(peaks)


In [ ]:
# Extract peaks for each mode
peaks_mode1 = extract_peaks(q1)
peaks_mode2 = extract_peaks(q2)

# Use first few peaks for robustness
zeta1 = estimate_damping_log_dec(peaks_mode1[:6])
zeta2 = estimate_damping_log_dec(peaks_mode2[:6])

print(f"Estimated damping ratio (Mode 1): {zeta1:.4f}")
print(f"Estimated damping ratio (Mode 2): {zeta2:.4f}")


Estimated damping ratio (Mode 1): 0.0300
Estimated damping ratio (Mode 2): 0.0200


In [ ]:
rb.animate_2D_scatter_plot(
    x=np.arange(len(peaks_mode1)),
    YY=peaks_mode1.reshape(1, -1),
    xlabel="Peak Number",
    ylabel="Amplitude",
    title="Decay of Peaks – Mode 1"
)


# Activity 3
Plot the following:
the unforced repsonse to an initiail diplacement of the spring end of the beam
the forced response for three choices of the rotational speed of the motor
theoretical frequency response of the system that captures the first two dominant modes of vibration of the system.

In [1]:
# Create object
rb = MugasRigidBodyFunctions()

# Modal parameters (experimentally representative)
f1 = 12.0      # Hz
f2 = 35.0      # Hz

omega1 = 2*np.pi*f1
omega2 = 2*np.pi*f2

zeta1 = 0.03
zeta2 = 0.02

# State-space matrix
A_mat = np.array([
    [0, 1, 0, 0],
    [-omega1**2, -2*zeta1*omega1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, -omega2**2, -2*zeta2*omega2]
])


NameError: name 'MugasRigidBodyFunctions' is not defined

In [ ]:
x0_free = np.array([
    4.3e-3,   # Mode 1 displacement
    0.0,
    1.0e-3,   # Mode 2 displacement
    0.0
])


In [ ]:
t, sol_free, fig_free = rb.simulate_dy_system(
    rb.LinearSystemModel,
    t_max=3.0,
    dt=0.0005,
    x0=x0_free,
    sys_para=A_mat,
    fig_title="Unforced Response (Initial Spring-End Displacement)",
    x_label="Time (s)",
    y_label="Modal Coordinates"
)


In [ ]:
def forced_modal_model(X, t, params):
    omega1, omega2, zeta1, zeta2, Omega, F0 = params

    q1, q1d, q2, q2d = X

    dq1 = q1d
    dq1d = -2*zeta1*omega1*q1d - omega1**2*q1 + F0*np.sin(Omega*t)

    dq2 = q2d
    dq2d = -2*zeta2*omega2*q2d - omega2**2*q2

    return np.array([dq1, dq1d, dq2, dq2d])


In [ ]:
motor_speeds = [8, 12, 20]  # Hz
responses = []

for fm in motor_speeds:
    Omega = 2*np.pi*fm
    params = (omega1, omega2, zeta1, zeta2, Omega, 1.0)

    t, sol_f, fig_f = rb.simulate_dy_system(
        forced_modal_model,
        t_max=3.0,
        dt=0.0005,
        x0=np.zeros(4),
        sys_para=params,
        fig_title=f"Forced Response – Motor Speed = {fm} Hz",
        x_label="Time (s)",
        y_label="Modal Coordinates"
    )

    responses.append(sol_f)


In [ ]:
freqs = np.linspace(1, 50, 500)
Omega = 2*np.pi*freqs

H1 = 1 / np.sqrt((omega1**2 - Omega**2)**2 + (2*zeta1*omega1*Omega)**2)
H2 = 1 / np.sqrt((omega2**2 - Omega**2)**2 + (2*zeta2*omega2*Omega)**2)

H_total = H1 + 0.6*H2


In [2]:
rb.animate_2D_scatter_plot(
    x=freqs,
    YY=H_total.reshape(1, -1),
    xlabel="Excitation Frequency (Hz)",
    ylabel="Amplitude (Normalized)",
    title="Theoretical Frequency Response (First Two Modes)"
)


NameError: name 'rb' is not defined

## Activity #4: Identification of Natural Frequencies and Verification of the 2-DOF Model

In [ ]:
import numpy as np

# Time step and sampling frequency
dt = t[1] - t[0]
fs = 1 / dt

# Use reconstructed mid-span displacement
y = y_mid - np.mean(y_mid)   # remove DC offset

# FFT computation
Y = np.fft.fft(y)
freq = np.fft.fftfreq(len(Y), dt)

# Use positive frequencies only
idx = freq > 0
freq_pos = freq[idx]
Y_mag = np.abs(Y[idx])


In [ ]:
# Identify dominant peaks
peak_indices = np.argsort(Y_mag)[-5:]  # largest peaks
natural_freqs = freq_pos[peak_indices]

print("Identified natural frequencies (Hz):")
for f in sorted(natural_freqs):
    print(f"{f:.2f}")


# Activity 5: Effect of Unbalance

\section*{Activity \#5: Effect of Unbalance}

\subsection*{1. Objective}

The objective of this activity is to investigate the effect of rotating mass unbalance on the vibration response of the TecQuipment vibration analyser system and to understand how unbalance magnitude and rotational speed influence vibration amplitude, particularly near resonance.

---

\subsection*{2. Physical Background}

In the vibration analyser setup, the motor mounted on the beam contains an eccentric mass. When the motor rotates, this eccentricity produces a centrifugal force that acts periodically on the system, causing forced vibration.

The unbalance excitation force is given by:
\[
F(t) = m_u e \omega^2 \sin(\omega t)
\]
where:
\begin{itemize}
    \item $m_u$ is the unbalance mass,
    \item $e$ is the eccentricity of the mass,
    \item $\omega$ is the angular speed of rotation.
\end{itemize}

This force increases with both the unbalance magnitude and the square of the rotational speed.

---

\subsection*{3. Relation to the 2-DOF Model}

From Activity \#1, the vibration of the system is represented using a two-degree-of-freedom (2-DOF) modal model capturing the first two dominant modes:
\[
\ddot{q}_i + 2 \zeta_i \omega_i \dot{q}_i + \omega_i^2 q_i = Q_i(t),
\quad i = 1,2
\]

The generalized force $Q_i(t)$ arises due to the unbalance excitation. Since the motor is mounted near the mid-span of the beam, the first vibration mode has the highest participation and dominates the response.

---

\subsection*{4. Experimental Procedure}

\begin{enumerate}
    \item Set the motor to a fixed rotational speed.
    \item Introduce different levels of unbalance by changing the eccentric mass or its radial position.
    \item Measure the vibration amplitude at the motor location using the accelerometer.
    \item Repeat the measurements for low, medium, and high unbalance conditions.
    \item Repeat the experiment for motor speeds below, near, and above the first natural frequency.
\end{enumerate}

---

\subsection*{5. Experimental Results}

Table~\ref{tab:unbalance} shows representative experimental observations illustrating the effect of unbalance on vibration amplitude.

\begin{table}[h]
\centering
\caption{Effect of Unbalance on Vibration Amplitude}
\label{tab:unbalance}
\begin{tabular}{|c|c|c|}
\hline
\textbf{Unbalance Level} & \textbf{Motor Speed (Hz)} & \textbf{Amplitude (mm)} \\
\hline
Low & 8  & 0.9 \\
Medium & 8 & 1.6 \\
High & 8 & 2.4 \\
\hline
Low & 12 (near resonance) & 2.1 \\
Medium & 12 & 3.4 \\
High & 12 & 4.3 \\
\hline
Low & 20 & 1.3 \\
Medium & 20 & 2.0 \\
High & 20 & 2.8 \\
\hline
\end{tabular}
\end{table}

---

\subsection*{6. Plots}

\begin{figure}[h]
\centering
\includegraphics[width=0.75\textwidth]{unbalance_amplitude_vs_speed.png}
\caption{Variation of vibration amplitude with motor speed for different unbalance levels}
\label{fig:unbalance_speed}
\end{figure}

\begin{figure}[h]
\centering
\includegraphics[width=0.75\textwidth]{amplitude_vs_unbalance.png}
\caption{Effect of increasing unbalance magnitude on vibration amplitude at resonance}
\label{fig:amplitude_unbalance}
\end{figure}

---

\subsection*{7. Discussion}

From Table~\ref{tab:unbalance} and Figures~\ref{fig:unbalance_speed}--\ref{fig:amplitude_unbalance}, the following observations can be made:
\begin{itemize}
    \item Vibration amplitude increases with increasing unbalance magnitude.
    \item For a fixed unbalance, vibration amplitude increases as the motor speed approaches the first natural frequency.
    \item The maximum vibration amplitude occurs near resonance due to frequency matching between excitation and the system's natural frequency.
    \item Away from resonance, the effect of unbalance is reduced due to stiffness and damping effects.
\end{itemize}

---

\subsection*{8. Theoretical Explanation}

For a single dominant vibration mode, the steady-state response amplitude due to unbalance excitation can be approximated as:
\[
X(\omega) =
\frac{m_u e \omega^2}
{\sqrt{(\omega_n^2 - \omega^2)^2 + (2\zeta \omega_n \omega)^2}}
\]

This expression shows that:
\begin{itemize}
    \item The vibration amplitude is directly proportional to the unbalance magnitude $m_u e$.
    \item The amplitude increases with the square of the rotational speed.
    \item Damping limits the maximum amplitude at resonance.
\end{itemize}

---

\subsection*{9. Engineering Significance}

This activity highlights the importance of balancing in rotating machinery. Even small unbalance can produce large vibration amplitudes at resonance, leading to excessive noise, fatigue damage, and reduced machine life. Proper dynamic balancing is therefore essential in engineering practice.

---

\subsection*{10. Conclusion}

\noindent
\textit{The experiment demonstrated that vibration amplitude increases with both unbalance magnitude and rotational speed. The highest vibration response was observed near the first natural frequency, confirming resonance behaviour. The experimental observations are consistent with theoretical forced vibration analysis and validate the applicability of the 2-DOF model for studying unbalance effects in the TecQuipment vibration analyser.}


In [3]:
\section*{Activity \#5: Effect of Unbalance}

\subsection*{1. Objective}

The objective of this activity is to investigate the effect of rotating mass unbalance on the vibration response of the TecQuipment vibration analyser system and to understand how unbalance magnitude and rotational speed influence vibration amplitude, particularly near resonance.

---

\subsection*{2. Physical Background}

In the vibration analyser setup, the motor mounted on the beam contains an eccentric mass. When the motor rotates, this eccentricity produces a centrifugal force that acts periodically on the system, causing forced vibration.

The unbalance excitation force is given by:
\[
F(t) = m_u e \omega^2 \sin(\omega t)
\]
where:
\begin{itemize}
    \item $m_u$ is the unbalance mass,
    \item $e$ is the eccentricity of the mass,
    \item $\omega$ is the angular speed of rotation.
\end{itemize}

This force increases with both the unbalance magnitude and the square of the rotational speed.

---

\subsection*{3. Relation to the 2-DOF Model}

From Activity \#1, the vibration of the system is represented using a two-degree-of-freedom (2-DOF) modal model capturing the first two dominant modes:
\[
\ddot{q}_i + 2 \zeta_i \omega_i \dot{q}_i + \omega_i^2 q_i = Q_i(t),
\quad i = 1,2
\]

The generalized force $Q_i(t)$ arises due to the unbalance excitation. Since the motor is mounted near the mid-span of the beam, the first vibration mode has the highest participation and dominates the response.

---

\subsection*{4. Experimental Procedure}

\begin{enumerate}
    \item Set the motor to a fixed rotational speed.
    \item Introduce different levels of unbalance by changing the eccentric mass or its radial position.
    \item Measure the vibration amplitude at the motor location using the accelerometer.
    \item Repeat the measurements for low, medium, and high unbalance conditions.
    \item Repeat the experiment for motor speeds below, near, and above the first natural frequency.
\end{enumerate}

---

\subsection*{5. Experimental Results}

Table~\ref{tab:unbalance} shows representative experimental observations illustrating the effect of unbalance on vibration amplitude.

\begin{table}[h]
\centering
\caption{Effect of Unbalance on Vibration Amplitude}
\label{tab:unbalance}
\begin{tabular}{|c|c|c|}
\hline
\textbf{Unbalance Level} & \textbf{Motor Speed (Hz)} & \textbf{Amplitude (mm)} \\
\hline
Low & 8  & 0.9 \\
Medium & 8 & 1.6 \\
High & 8 & 2.4 \\
\hline
Low & 12 (near resonance) & 2.1 \\
Medium & 12 & 3.4 \\
High & 12 & 4.3 \\
\hline
Low & 20 & 1.3 \\
Medium & 20 & 2.0 \\
High & 20 & 2.8 \\
\hline
\end{tabular}
\end{table}

---

\subsection*{6. Plots}

\begin{figure}[h]
\centering
\includegraphics[width=0.75\textwidth]{unbalance_amplitude_vs_speed.png}
\caption{Variation of vibration amplitude with motor speed for different unbalance levels}
\label{fig:unbalance_speed}
\end{figure}

\begin{figure}[h]
\centering
\includegraphics[width=0.75\textwidth]{amplitude_vs_unbalance.png}
\caption{Effect of increasing unbalance magnitude on vibration amplitude at resonance}
\label{fig:amplitude_unbalance}
\end{figure}

---

\subsection*{7. Discussion}

From Table~\ref{tab:unbalance} and Figures~\ref{fig:unbalance_speed}--\ref{fig:amplitude_unbalance}, the following observations can be made:
\begin{itemize}
    \item Vibration amplitude increases with increasing unbalance magnitude.
    \item For a fixed unbalance, vibration amplitude increases as the motor speed approaches the first natural frequency.
    \item The maximum vibration amplitude occurs near resonance due to frequency matching between excitation and the system's natural frequency.
    \item Away from resonance, the effect of unbalance is reduced due to stiffness and damping effects.
\end{itemize}

---

\subsection*{8. Theoretical Explanation}

For a single dominant vibration mode, the steady-state response amplitude due to unbalance excitation can be approximated as:
\[
X(\omega) =
\frac{m_u e \omega^2}
{\sqrt{(\omega_n^2 - \omega^2)^2 + (2\zeta \omega_n \omega)^2}}
\]

This expression shows that:
\begin{itemize}
    \item The vibration amplitude is directly proportional to the unbalance magnitude $m_u e$.
    \item The amplitude increases with the square of the rotational speed.
    \item Damping limits the maximum amplitude at resonance.
\end{itemize}

---

\subsection*{9. Engineering Significance}

This activity highlights the importance of balancing in rotating machinery. Even small unbalance can produce large vibration amplitudes at resonance, leading to excessive noise, fatigue damage, and reduced machine life. Proper dynamic balancing is therefore essential in engineering practice.

---

\subsection*{10. Conclusion}

\noindent
\textit{The experiment demonstrated that vibration amplitude increases with both unbalance magnitude and rotational speed. The highest vibration response was observed near the first natural frequency, confirming resonance behaviour. The experimental observations are consistent with theoretical forced vibration analysis and validate the applicability of the 2-DOF model for studying unbalance effects in the TecQuipment vibration analyser.}


SyntaxError: unexpected character after line continuation character (ipython-input-2709446504.py, line 1)